## BMLL Notebook Educational Series
### Finding a Signal in the Noise: Order Book Analysis

In [1]:
import os
from functools import lru_cache
import sys

def install_snowflake_connector():
    os.system(sys.executable + " -m pip install snowflake-connector-python[pandas] snowflake-sqlalchemy")

@lru_cache(5)
def snowflake_connection(
    database="PROD_DERIVED_DATA_DB",
    schema="PUBLIC",
    warehouse="PROD_DERIVED_DEFAULT_WH",
    role="PROD_SALES_RO_ROLE",
    user=None,
    password=None,
    use_secret_storage=True,
    sso=None,
):
    import snowflake
    import snowflake.connector
    try:
        from snowflake.snowpark import Session as SnowparkSession
    except Exception:
        SnowparkSession = None
    try:
        import bmll2
        stor = bmll2.storage_paths()["user"]
        os.environ["SNOWFLAKE_USER"] = stor["prefix"].split("/")[1]
        os.environ["SNOWFLAKE_USE_SSO"] = "1"
    except Exception:
        pass
    if sso is None:
        sso = os.environ.get("SNOWFLAKE_USE_SSO")

    password = os.environ.get("SNOWFLAKE_PASSWORD", password)
    if not password:
        try:
            import secretstorage
            connection = secretstorage.dbus_init()
            collection = secretstorage.get_default_collection(connection)
            for item in collection.search_items({"name": "default_snowflake", "system": "bmll"}):
                password = item.get_secret()
        except Exception as e:
            print(f"Error connecting to Secret Service: {e}")
    else:
        try:
            import secretstorage
            attributes = {"name": "default_snowflake", "system": "bmll"}
            label = "default_snowflake"
            secret = password
            connection = secretstorage.dbus_init()
            collection = secretstorage.get_default_collection(connection)
            if collection.is_locked():
                unlocked = collection.unlock()
                if unlocked:
                    print("Collection unlocked successfully.")
                else:
                    print("Failed to unlock the collection.")
            else:
                print("Collection is already unlocked.")
            collection.create_item(label, attributes, secret, replace=True)
        except Exception as e:
            print(e)

    if not sso:
        connargs = dict(
            user=os.environ.get("SNOWFLAKE_USER", user),
            password=password,
            account=os.environ.get("SNOWFLAKE_ACCOUNT", "tib96089"),
            region=os.environ.get("SNOWFLAKE_REGION", "us-east-1"),
            role=role,
            database=database,
            schema=schema,
            warehouse=warehouse,
        )
    else:
        connargs = dict(
            user=os.environ.get("SNOWFLAKE_USER", user),
            account=os.environ.get("SNOWFLAKE_ACCOUNT", "tib96089"),
            region=os.environ.get("SNOWFLAKE_REGION", "us-east-1"),
            role=role,
            database=database,
            schema=schema,
            warehouse=warehouse,
            authenticator="externalbrowser",
        )
    if SnowparkSession is not None:
        return SnowparkSession.builder.configs(connargs).create()
    return snowflake.connector.connect(**connargs)

session = snowflake_connection()
if not hasattr(session, "sql"):
    class _SFResult:
        def __init__(self, cursor):
            self._cursor = cursor

        def to_pandas(self):
            try:
                return self._cursor.fetch_pandas_all()
            except Exception:
                import pandas as pd
                rows = self._cursor.fetchall()
                cols = [c[0] for c in self._cursor.description] if self._cursor.description else []
                return pd.DataFrame(rows, columns=cols)

    class _SFSessionShim:
        def __init__(self, conn):
            self._conn = conn

        def sql(self, query):
            cur = self._conn.cursor()
            cur.execute(query)
            return _SFResult(cur)

    session = _SFSessionShim(session)

/home/bertrand/miniconda3/lib/python3.11/site-packages/bmll2/_internals/api/configure.py:116: UserWarning: BMLL configuration file (/tmp/.nb_extension_instance_vars.json) not found. BMLL data and S3 storage will not be reachable.
  warnings.warn(f"{msg} BMLL data and S3 storage will not be reachable.", UserWarning)


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://accounts.google.com/o/saml2/idp?idpid=C0433grw8&SAMLRequest=lZLLbtswEEV%2FRWDXEin5TVg2HLuBXcSJ60cX3dHiWGZNkSpJRXG%2FvpQfQLpIgC4kCNS9PDNzZzh%2BK2TwCsYKrVIURwQFoDLNhcpTtNs%2Bhn0UWMcUZ1IrSNEZLBqPhpYVsqSTyh3VGn5XYF3gL1KWNj9SVBlFNbPCUsUKsNRldDNZPtEkIpRZC8Z5HLpZuBWedXSupBjXdR3VrUibHCeEEEwG2KsayRf0DlF%2BziiNdjrT8m558z19gIgxaTcIr%2FCE1c34INR1BJ9R9leRpfPtdhWuXjZbFEzu3U21slUBZgPmVWSwWz9dC7C%2Bgn0hZdi8HGTHyCpdHyQ7QaaLsnL%2Bxsh%2F4QNwLHUu%2FJwWsxSVJ8GP06%2BndtIDfpaDes8eBv31y3JnkvmyN%2F%2F2vct6%2B1%2BuUH9iuc4zFPy4p5o0qS6srWChmiydPyJJNyT9kPS2cYeSFu10om6%2F%2FRMFM5%2BlUMxdnPeCWZbpSjkb5VrnEi71adwEkWDBy7F%2FBE%2BnpN1q5abuo%2BuC0AvTjP6n7SF%2B77zt2bMf%2FWK20lJk5%2BBRm4K5j5OJo%2FhyInh4uEgpFEzICecGrPUJSanrqQHm%2FDo7UwHCoyv134Ue%2FQU%3D&RelayState=ver%3A3

In [2]:
start_date = '2023-10-02'
end_date = '2023-10-05'

### Signal Discovery from Market Microstructure Data

**Objective**: Discover alpha-generating trading signals by analyzing order book dynamics.
The fundamental hypothesis is that **price movements are predictable from order book characteristics**.

**Why Order Book Data?**
- Captures market microstructure (limit orders, cancellations, executions)
- Reveals trader intent before prices move
- Enables early detection of momentum and reversals
- Shows supply/demand imbalances at different price levels

**Signal Types** that can be extracted from Level 2/3 order book:
- **Imbalance signals**: Significant bid/ask size differences predict price movement
- **Depth signals**: Shallow book depth often precedes large moves
- **Order flow signals**: Clustering of orders at certain levels indicates support/resistance
- **Momentum signals**: Rate of change in best bid/ask often leads price movement

This notebook uses statistical regression to measure correlations between order book metrics
and subsequent price changes (alpha) across the S&P 500 universe in Snowflake.

## Import Libraries

In [3]:
try:
    import statsmodels
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "statsmodels"])

import pytz
import pandas as pd
import statsmodels.api as sm

from datetime import datetime, timedelta

In [4]:
# Expect an existing Snowpark session named `session` in Snowflake worksheets.

## Phase 1: Data Retrieval and Preparation

This section gathers Level 3 order book data (10 levels deep) and preprocesses it for analysis.
The process:
1. **Universe definition**: Select S&P 500 securities traded on US venues
2. **Reference mapping**: Map tickers to BMLL Listing IDs
3. **Data fetching**: Retrieve Level 2/3 order book snapshots at regular intervals
4. **Feature engineering**: Extract imbalance and depth metrics from order book

Level 2 data provides bid/ask prices and quantities at up to 10 price levels.
Level 3 data shows individual orders (more granular but more computation-intensive).

### Data Collection: Level 3 Order Book

### Select Universe (Top 500 stocks)

### Get ListingIds (BMLL Ticker-MIC ids that map to a ticker and venue)

In [5]:
# Define list of US market MIC codes to filter by
us_mics = ['XNAS', 'XBOS', 'XPSX', 'XNYS', 'XASE', 'ARCX', 'BATY',
           'BATS', 'EDGA', 'EDGX', 'XCIS', 'XCHI', 'IEXG', 'MEMX', 'EPRL']

In [6]:
mic_list = ", ".join(f"'{mic}'" for mic in us_mics)
ref_sql = f"""
    SELECT DISTINCT TICKER AS "Ticker"
    FROM "PROD_DERIVED_DATA_DB"."PUBLIC"."REFERENCE"
    WHERE IS_ALIVE = TRUE
      AND IS_PRIMARY = TRUE
      AND MIC IN ({mic_list})
    LIMIT 500
"""
universe = session.sql(ref_sql).to_pandas()
universe_symbols = universe["Ticker"].tolist()

ref_lid_sql = f"""
    SELECT LISTING_ID AS "ListingId"
    FROM "PROD_DERIVED_DATA_DB"."PUBLIC"."REFERENCE"
    WHERE IS_ALIVE = TRUE
      AND IS_PRIMARY = TRUE
      AND MIC IN ({mic_list})
      AND TICKER IN ({", ".join(f"'{t}'" for t in universe_symbols)})
"""
universe_lids = session.sql(ref_lid_sql).to_pandas()["ListingId"].tolist()

### Time Period and other variables

In [7]:
start_date = "2023-10-02"
end_date = "2023-10-05"
freq = 60 ## interval size

# Convert start_date and end_date strings to datetime objects
start_date_obj = datetime.strptime(start_date, "%Y-%m-%d")
end_date_obj = datetime.strptime(end_date, "%Y-%m-%d")

# Generate a list of dates between start_date and end_date (inclusive)
dates = [start_date_obj + timedelta(days=x) for x in range((end_date_obj - start_date_obj).days + 1)]

# Convert date objects back to strings
dates = [date.strftime("%Y-%m-%d") for date in dates]

In [8]:
start_time = "09:30"
end_time = "16:00"

### Collect L3 Data

In [9]:
def _utc_bounds(date, start_time, end_time):
    ts_start = pd.Timestamp(f"{date} {start_time}").tz_localize("US/Eastern").tz_convert("UTC").tz_localize(None)
    ts_end = pd.Timestamp(f"{date} {end_time}").tz_localize("US/Eastern").tz_convert("UTC").tz_localize(None)
    return ts_start, ts_end

In [10]:
def get_L2_data(lid, date, start_time, end_time, freq=60, depth=10):
    ts_start, ts_end = _utc_bounds(date, start_time, end_time)
    level_cols = []
    for i in range(1, depth + 1):
        level_cols.extend([
            f"BID_PRICE{i}", f"BID_QUANTITY{i}",
            f"ASK_PRICE{i}", f"ASK_QUANTITY{i}",
        ])
    cols_sql = ", ".join(["EVENT_TIMESTAMP", "BMLL_OBJECT_ID AS LISTING_ID"] + level_cols)
    sql = f"""
        SELECT {cols_sql}
        FROM "PROD_L2_CBBO_DB"."PUBLIC"."VW_L2_DATA_EQUITY"
        WHERE BMLL_OBJECT_ID = {lid}
          AND TRADE_DATE = '{date}'
          AND EVENT_TIMESTAMP BETWEEN '{ts_start}' AND '{ts_end}'
        ORDER BY EVENT_TIMESTAMP
    """
    df = session.sql(sql).to_pandas()
    if df.empty:
        return pd.DataFrame(columns=["timestamp", "lid", "side", "price", "size"])
    df["EVENT_TIMESTAMP"] = pd.to_datetime(df["EVENT_TIMESTAMP"])
    df = df.sort_values("EVENT_TIMESTAMP")
    df = (
        df.groupby("LISTING_ID", group_keys=False)
        .resample(f"{freq}S", on="EVENT_TIMESTAMP")
        .last()
        .reset_index()
    )

    records = []
    for _, row in df.iterrows():
        ts = row["EVENT_TIMESTAMP"]
        lid_val = row["LISTING_ID"]
        for i in range(1, depth + 1):
            bid_p = row.get(f"BID_PRICE{i}")
            bid_q = row.get(f"BID_QUANTITY{i}")
            ask_p = row.get(f"ASK_PRICE{i}")
            ask_q = row.get(f"ASK_QUANTITY{i}")
            if pd.notna(bid_p) and pd.notna(bid_q):
                records.append((ts, lid_val, 1, bid_p, bid_q))
            if pd.notna(ask_p) and pd.notna(ask_q):
                records.append((ts, lid_val, 2, ask_p, ask_q))
    return pd.DataFrame(records, columns=["timestamp", "lid", "side", "price", "size"])

In [11]:
l3 = get_L2_data(322753, "2023-10-02", start_time, end_time, freq=freq, depth=10)

/tmp/ipykernel_659623/3689999146.py:25: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  .resample(f"{freq}S", on="EVENT_TIMESTAMP")


## Phase 2: Feature Engineering and Signal Extraction

### Signal Metrics for Predictiveness

We extract three key order book imbalance metrics at fixed time intervals. These ratios measure
buy-side vs sell-side characteristics, which often precede price movements:

**1. Notional Imbalance Ratio**
- Ratio of total buy-side volume to sell-side volume
- **Interpretation**: Values > 1 suggest more buying pressure (bullish)
- **Signal**: Large ratios often precede upward price movements

**2. Average Size Ratio**
- Ratio of average buy order size to average sell order size
- **Interpretation**: Larger buy orders suggest institutional buying
- **Signal**: Disparity in order sizes can indicate informed trading

**3. Order Count Ratio**
- Ratio of number of buy orders to sell orders
- **Interpretation**: More buy orders may indicate bid-side support
- **Signal**: Order count alone has weaker predictive power but useful in combination

**Why These Metrics?**
The hypothesis is that market microstructure (how orders are placed, not just their prices)
contains information about future price movements. By analyzing these ratios at regular intervals,
we can identify patterns that precede directional moves.

### Calculate Imbalance Ratios at Each Time Snapshot

In [12]:
# Step 1: Separate buy-side (side=1) and sell-side (side=2) orders
side1_data = l3[l3['side'] == 1]
side2_data = l3[l3['side'] == 2]

# Step 2: Group by timestamp and security (lid) to analyze book state at each snapshot
grouped_side1 = side1_data.groupby(['timestamp', 'lid'])
grouped_side2 = side2_data.groupby(['timestamp', 'lid'])

# Step 3: Calculate three aggregate metrics per side
# - sum: total notional volume (sum of all sizes)
# - mean: average order size per level
# - size: order count
sum_notional_side1 = grouped_side1['size'].sum()
avg_size_side1 = grouped_side1['size'].mean()
count_side1 = grouped_side1.size()

sum_notional_side2 = grouped_side2['size'].sum()
avg_size_side2 = grouped_side2['size'].mean()
count_side2 = grouped_side2.size()

# Step 4: Compute ratios (buy-side / sell-side)
# Ratios > 1 indicate buy-side dominance
sum_notional_ratio = sum_notional_side1 / sum_notional_side2
avg_size_ratio = avg_size_side1 / avg_size_side2
count_ratio = count_side1 / count_side2

# Step 5: Consolidate all ratios into a single DataFrame for regression
ratios_df = pd.DataFrame({
    'timestamp': sum_notional_ratio.index.get_level_values('timestamp'),
    'lid': sum_notional_ratio.index.get_level_values('lid'),
    'sum_notional_ratio': sum_notional_ratio.values,
    'avg_size_ratio': avg_size_ratio.values,
    'count_ratio': count_ratio.values
})

ratios_df

,timestamp,lid,sum_notional_ratio,avg_size_ratio,count_ratio
0,2023-10-02 13:30:00,322753,1.115229,1.115229,1.0
1,2023-10-02 13:31:00,322753,0.492379,0.492379,1.0
2,2023-10-02 13:32:00,322753,0.800399,0.800399,1.0
3,2023-10-02 13:33:00,322753,0.568920,0.568920,1.0
4,2023-10-02 13:34:00,322753,0.840922,0.840922,1.0
...,...,...,...,...,...
385,2023-10-02 19:55:00,322753,0.838399,0.838399,1.0
386,2023-10-02 19:56:00,322753,0.904475,0.904475,1.0
387,2023-10-02 19:57:00,322753,0.602841,0.602841,1.0
388,2023-10-02 19:58:00,322753,0.839105,0.839105,1.0


### Calculate CBBO at Each Snapshot

We compute the consolidated best bid/offer (CBBO) by finding:
- **Best bid**: Maximum bid price across all levels (buy-side strength)
- **Best ask**: Minimum ask price across all levels (sell-side strength)
- **Mid price**: Average of best bid and ask (fair value benchmark)

The mid price serves as the dependent variable in our regression - we want to know if
the order book imbalance ratios can predict future mid prices.

In [13]:
# Step 1: Ensure timestamps are in datetime format
l3['timestamp'] = pd.to_datetime(l3['timestamp'])

# Step 2: Separate buy and sell side orders
side1_prices = l3[l3['side'] == 1]  # Buy side (bids)
side2_prices = l3[l3['side'] == 2]  # Sell side (asks)

# Step 3: Find best bid (max buy price) and best ask (min sell price) at each snapshot
# These represent the CBBO (consolidated best bid/offer)
side1_max_prices = side1_prices.groupby(['timestamp', 'lid'])['price'].max().reset_index()
side2_min_prices = side2_prices.groupby(['timestamp', 'lid'])['price'].min().reset_index()

# Step 4: Merge bid and ask prices
final_prices = pd.merge(side1_max_prices, side2_min_prices, on=['timestamp', 'lid'], suffixes=('_side1', '_side2'))

# Step 5: Calculate mid price as the average of bid and ask
final_prices['mid_price'] = (final_prices['price_side2'] + final_prices['price_side1']) / 2

# Step 6: Merge imbalance ratios with prices for regression analysis
ratio_mid = ratios_df.merge(final_prices)
ratio_mid

,timestamp,lid,sum_notional_ratio,avg_size_ratio,count_ratio,price_side1,price_side2,mid_price
0,2023-10-02 13:30:00,322753,1.115229,1.115229,1.0,40.91,41.00,40.955
1,2023-10-02 13:31:00,322753,0.492379,0.492379,1.0,40.80,40.84,40.820
2,2023-10-02 13:32:00,322753,0.800399,0.800399,1.0,40.84,40.90,40.870
3,2023-10-02 13:33:00,322753,0.568920,0.568920,1.0,40.80,40.86,40.830
4,2023-10-02 13:34:00,322753,0.840922,0.840922,1.0,40.72,40.77,40.745
...,...,...,...,...,...,...,...,...
385,2023-10-02 19:55:00,322753,0.838399,0.838399,1.0,39.45,39.46,39.455
386,2023-10-02 19:56:00,322753,0.904475,0.904475,1.0,39.47,39.48,39.475
387,2023-10-02 19:57:00,322753,0.602841,0.602841,1.0,39.49,39.50,39.495
388,2023-10-02 19:58:00,322753,0.839105,0.839105,1.0,39.49,39.50,39.495


## Phase 3: Statistical Analysis and Hypothesis Testing

Now we test whether the order book imbalance metrics have predictive power for price movements.
The analysis includes:
1. **Correlation analysis**: Measure linear relationship between signals and prices
2. **Regression modeling**: Quantify the impact of each signal
3. **Statistical significance**: Determine if observed relationships are meaningful

### Correlation Analysis: Signal Strength

We first compute simple correlations between each imbalance metric and price.
Strong correlations suggest the metric has predictive value, though correlation
alone does not establish causality.

In [14]:
# Step 1: Calculate pairwise correlation between each imbalance metric and mid price
# Correlation ranges from -1 (inverse) to +1 (positive relationship)
# A value close to 0 suggests no linear relationship
correlation = ratio_mid['sum_notional_ratio'].corr(ratio_mid['mid_price'])
print("Correlation between 'notional_ratio' and 'mid_price':", correlation)

Correlation between 'notional_ratio' and 'mid_price': 0.03617174436972724


In [15]:
# ## Regression Analysis: Quantifying Predictive Power
#
# ### Model Specification
#
# We use Ordinary Least Squares (OLS) regression to estimate the relationship:
# **mid_price = β₀ + β₁(notional_ratio) + β₂(size_ratio) + β₃(count_ratio) + ε**
#
# **Interpretation of coefficients:**
# - **β₀ (intercept)**: Base price level
# - **β₁-β₃**: Change in price for a unit change in each ratio (holding others constant)
# - **P-values**: Probability that the coefficient is zero (< 0.05 suggests significance)
# - **R²**: Proportion of price variance explained by the model

In [16]:
# Step 1: Define dependent variable (what we want to predict)
y = ratio_mid['mid_price']

# Step 2: Define independent variables (what we use to predict)
X = ratio_mid[['sum_notional_ratio', 'avg_size_ratio', 'count_ratio']]

# Step 3: Add intercept term (constant) for regression
X = sm.add_constant(X)

# Step 4: Fit OLS regression model
model = sm.OLS(y, X).fit()

# Step 5: Display comprehensive regression results
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              mid_price   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                    0.5083
Date:                Fri, 07 Aug 2026   Prob (F-statistic):              0.476
Time:                        16:07:00   Log-Likelihood:                -88.845
No. Observations:                 390   AIC:                             181.7
Df Residuals:                     388   BIC:                             189.6
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
sum_notional_ratio     0.0148      0

In [17]:
# If it's not already, make sure 'lid' is of type category for regression analysis

# Assuming ratio_mid is your DataFrame
# Convert 'lid' to categorical type
ratio_mid['lid'] = ratio_mid['lid'].astype('category')

# Define independent variables
X = ratio_mid[['sum_notional_ratio', 'avg_size_ratio', 'count_ratio']]

# Add constant term to the independent variables
X = sm.add_constant(X)

# Define dependent variable
y = ratio_mid['mid_price']

# Perform regression for each lid
results = {}
for lid, group in ratio_mid.groupby('lid'):
    lid_str = str(lid)  # Convert lid to string for use as dictionary key
    X_group = X[ratio_mid['lid'] == lid]  # Subset X for current lid
    y_group = y[ratio_mid['lid'] == lid]  # Subset y for current lid
    
    # Fit OLS regression model
    model = sm.OLS(y_group, X_group)
    result = model.fit()
    results[lid_str] = result

# Display regression results
for lid, result in results.items():
    print(f"Regression results for lid {lid}:")
    print(result.summary())

Regression results for lid 322753:
                            OLS Regression Results                            
Dep. Variable:              mid_price   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                    0.5083
Date:                Fri, 07 Aug 2026   Prob (F-statistic):              0.476
Time:                        16:07:00   Log-Likelihood:                -88.845
No. Observations:                 390   AIC:                             181.7
Df Residuals:                     388   BIC:                             189.6
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
s

/tmp/ipykernel_659623/539267493.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for lid, group in ratio_mid.groupby('lid'):
